# 🔗 ImageBind — CSC14005 Final Project (Updated)
**Topic 17 | Nhóm 2 người | Focused type**

| Nội dung cập nhật | Chi tiết |
|------|----------|
| **Dataset** | ESC-50 (Kaggle path) & NYU Depth v2 (HuggingFace) |
| **Metrics** | Zero-shot Accuracy, Recall@1, R@5, R@10 |
| **Benchmarks** | Temperature, Epochs, Batch Size, Dropout (0, 0.1, 0.2) |
| **Demo** | Cross-modal Retrieval (Text → Audio, Audio → Image) |

> **Môi trường**: Kaggle GPU T4 x2 · Internet ON

## 1. Setup & Dependencies

In [ ]:
!git clone https://github.com/facebookresearch/ImageBind.git
%cd ImageBind
!pip install -q .
!pip install -q timm ftfy regex einops fvcore tqdm matplotlib seaborn pandas datasets torchaudio
print('\n=== Cài đặt hoàn tất ===')

In [ ]:
import os
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
from tqdm import tqdm
from imagebind import data as ibdata
from imagebind.models import imagebind_model
from imagebind.models.imagebind_model import ModalityType

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
model = imagebind_model.imagebind_huge(pretrained=True)
model.eval()
model.to(DEVICE)
print(f'Model loaded on {DEVICE}')

## 2. Dataset 1: ESC-50 (Audio Classification & Retrieval)

In [ ]:
import torchaudio
from torch.utils.data import Dataset, DataLoader

KAGGLE_ESC50_DIR = '/kaggle/input/datasets/hongngctng/dataesc50'
META_PATH = os.path.join(KAGGLE_ESC50_DIR, 'meta/esc50.csv')
AUDIO_DIR = os.path.join(KAGGLE_ESC50_DIR, 'audio')

class ESC50Dataset(Dataset):
    def __init__(self, meta_path, audio_dir, folds, target_sr=16000, duration=2.0):
        self.meta = pd.read_csv(meta_path)
        self.meta = self.meta[self.meta['fold'].isin(folds)].reset_index(drop=True)
        self.audio_dir = audio_dir
        self.target_sr = target_sr
        self.num_samples = int(target_sr * duration)
        categories = sorted(self.meta['category'].unique())
        self.class2idx = {c: i for i, c in enumerate(categories)}
        self.class_names = categories

    def __len__(self): return len(self.meta)

    def __getitem__(self, idx):
        row = self.meta.iloc[idx]
        waveform, sr = torchaudio.load(os.path.join(self.audio_dir, row['filename']))
        if sr != self.target_sr:
            waveform = torchaudio.transforms.Resample(sr, self.target_sr)(waveform)
        if waveform.shape[0] > 1: waveform = waveform.mean(dim=0, keepdim=True)
        if waveform.shape[1] < self.num_samples:
            waveform = F.pad(waveform, (0, self.num_samples - waveform.shape[1]))
        else: waveform = waveform[:, :self.num_samples]
        
        mel = torchaudio.transforms.MelSpectrogram(sample_rate=self.target_sr, n_fft=400, n_mels=128)(waveform)
        mel = torchaudio.transforms.AmplitudeToDB()(mel).squeeze(0)
        mel = (mel - mel.mean()) / (mel.std() + 1e-6)
        return mel, self.class2idx[row['category']]

test_dataset = ESC50Dataset(META_PATH, AUDIO_DIR, folds=[5])
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
print(f'ESC-50 Test samples: {len(test_dataset)}')

## 3. Benchmark Retrieval R@1, R@5, R@10

In [ ]:
@torch.no_grad()
def evaluate_retrieval(model, loader, class_names, device, temperature=0.05):
    model.eval()
    prompts = [f'This is a sound of {c.replace("_", " ")}' for c in class_names]
    text_inputs = {ModalityType.TEXT: ibdata.load_and_transform_text(prompts, device)}
    text_emb = F.normalize(model(text_inputs)[ModalityType.TEXT], dim=-1)

    all_preds, all_labels = [], []
    for mel_batch, label_batch in tqdm(loader, desc='Evaluating Retrieval'):
        audio_emb = F.normalize(model({ModalityType.AUDIO: mel_batch.unsqueeze(1).to(device)})[ModalityType.AUDIO], dim=-1)
        sim = audio_emb @ text_emb.T / temperature
        all_preds.append(sim.argsort(dim=-1, descending=True).cpu())
        all_labels.append(label_batch)

    all_preds = torch.cat(all_preds)
    all_labels = torch.cat(all_labels)
    
    results = {}
    for k in [1, 5, 10]:
        results[f'R@{k}'] = (all_preds[:, :k] == all_labels.view(-1, 1)).any(dim=1).float().mean().item() * 100
    return results

ret_results = evaluate_retrieval(model, test_loader, test_dataset.class_names, DEVICE)
for k, v in ret_results.items(): print(f'{k}: {v:.2f}%')

## 4. Dataset 2: NYU Depth v2 (Vision-Depth Alignment)

In [ ]:
from datasets import load_dataset
print("Loading NYU Depth V2 from HF...")
nyu_ds = load_dataset("sayakpaul/nyu_depth_v2", split="validation[:100]")

os.makedirs('nyu/img', exist_ok=True); os.makedirs('nyu/dep', exist_ok=True)
img_paths, dep_paths = [], []
for i, item in enumerate(nyu_ds):
    ip, dp = f'nyu/img/{i}.jpg', f'nyu/dep/{i}.png'
    item['image'].save(ip); item['depth_map'].save(dp)
    img_paths.append(ip); dep_paths.append(dp)

with torch.no_grad():
    inputs = {
        ModalityType.VISION: ibdata.load_and_transform_vision_data(img_paths, DEVICE),
        ModalityType.DEPTH: ibdata.load_and_transform_depth_data(dep_paths, DEVICE)
    }
    embs = model(inputs)
    v_emb, d_emb = F.normalize(embs[ModalityType.VISION], dim=-1), F.normalize(embs[ModalityType.DEPTH], dim=-1)
    
sim = d_emb @ v_emb.T
indices = sim.argsort(dim=-1, descending=True)
labels = torch.arange(len(img_paths)).to(DEVICE)
print("\n=== NYU Depth R@K (Depth -> Image) ===")
for k in [1, 5, 10]:
    acc = (indices[:, :k] == labels.view(-1, 1)).any(dim=1).float().mean().item() * 100
    print(f'R@{k}: {acc:.2f}%')

## 5. Benchmark Dropout (Audio Encoder)

In [ ]:
dropout_rates = [0.0, 0.1, 0.2]
print("=== Benchmark Audio Dropout ===")
for p in dropout_rates:
    do = torch.nn.Dropout(p=p)
    all_correct = 0
    # Tái sử dụng text_emb từ ESC-50
    for mel, lbl in test_loader:
        with torch.no_grad():
            a_emb = do(model({ModalityType.AUDIO: mel.unsqueeze(1).to(DEVICE)})[ModalityType.AUDIO])
            # So sánh đơn giản với class để đo độ nhạy
            all_correct += (F.normalize(a_emb, dim=-1) @ text_emb.T).argmax(dim=-1).cpu().eq(lbl).sum().item()
    print(f'Dropout {p}: Accuracy {all_correct/len(test_dataset)*100:.2f}%')

## 6. Demo Cross-modal Retrieval

In [ ]:
sample_idx = 0
a_path = os.path.join(AUDIO_DIR, test_dataset.meta.iloc[sample_idx]['filename'])
label = test_dataset.meta.iloc[sample_idx]['category']
texts = [f"A sound of {label}", "A photo of a forest", "Background noise"]

inputs = {
    ModalityType.AUDIO: ibdata.load_and_transform_audio_data([a_path], DEVICE),
    ModalityType.TEXT: ibdata.load_and_transform_text(texts, DEVICE),
    ModalityType.VISION: ibdata.load_and_transform_vision_data(img_paths[:3], DEVICE)
}

with torch.no_grad():
    e = model(inputs)
    t_e, a_e, v_e = F.normalize(e[ModalityType.TEXT], dim=-1), F.normalize(e[ModalityType.AUDIO], dim=-1), F.normalize(e[ModalityType.VISION], dim=-1)

print(f"Query Audio: {label}")
print(f"Best Text Match: {texts[(a_e @ t_e.T).argmax().item()]}")
print(f"Best Image Match (idx): {(a_e @ v_e.T).argmax().item()}")